In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader=PyPDFLoader(PDF_PATH)
pages =loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n","."," "],
)

chunks=splitter.split_documents(pages)
len(chunks)

37

In [7]:
chunks[0]

Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1')

In [8]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app'

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store= Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")


Loading weights: 100%|█| 103/103 [00:00<00:00, 3157.74


Vector store ready. 74 vectors stored.


In [12]:
retriever =vector_store.as_retriever(search_kwargs={"k":3})
test_query="What is VoLTE and how does it improve call quality?"
retrieved= retriever.invoke(test_query)

In [14]:
for i, doc in enumerate(retrieved,1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 3 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt



In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the  question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

llm=ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed",
)

chain=(
    {"context":retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("RAG chain assembled.")


RAG chain assembled.


In [19]:
question ="How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to a partner network in a foreign country when you're outside your home network's coverage. Here's how it works and the charges you might expect:

### **How It Works**  
1. **Connection**: Your device automatically links to a partner network in the visited country.  
2. **Authentication**: The visited network verifies your identity using signaling protocols (SS7 or Diameter).  
3. **Authorization**: Your home network checks your subscription and approves service.  
4. **Billing**: All data, voice, and SMS traffic is sent back to your home network for billing, which may introduce latency compared to local networks.  

### **Charges**  
Charges depend on the **roaming zone** of the country you're visiting:  
- **Zone A** (EU, UK, Australia, New Zealand): Lowest rates.  
- **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
- **Zone C** (Rest of the wo